# Core algorithms

Fayyad–Irani MDL discretisation, CFS with best-first search, the consistency measure with best-first search, Hidden (weight) naïve Bayes, naïve Bayes and correlation-weighted naïve Bayes, re-implemented following WEKA defaults. Other notebooks load this one with `%run core.ipynb`.

> Core algorithms: Fayyad-Irani MDL discretisation, CFS-BFS, CONSISTENCY-BFS, Hidden (weight) naive Bayes, naive Bayes and correlation-weighted naive Bayes. Re-implemented in Python following WEKA's defaults (Hall 1999; Dash & Liu 2003; Zhang, Jiang & Su 2005).

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import heapq
import numpy as np
from numba import njit

### MDL discretisation

In [ ]:
# ---------------------------------------------------------------- MDL discretisation
@njit
def _ent(counts, n):
    h = 0.0
    for c in counts:
        if c > 0:
            p = c / n
            h -= p * np.log2(p)
    return h

In [ ]:
@njit
def _mdl_rec(xs, ys, lo, hi, C, cuts, ncut):
    n = hi - lo
    if n < 2:
        return ncut
    tot = np.zeros(C)
    for i in range(lo, hi):
        tot[ys[i]] += 1
    hS = _ent(tot, n)
    k = 0
    for c in range(C):
        if tot[c] > 0:
            k += 1
    left = np.zeros(C)
    best_e = 1e18
    best_i = -1
    for i in range(lo, hi - 1):
        left[ys[i]] += 1
        if xs[i] == xs[i + 1]:
            continue
        nl = i - lo + 1
        nr = n - nl
        right = tot - left
        e = nl / n * _ent(left, nl) + nr / n * _ent(right, nr)
        if e < best_e:
            best_e = e
            best_i = i
    if best_i < 0:
        return ncut
    left = np.zeros(C)
    for i in range(lo, best_i + 1):
        left[ys[i]] += 1
    right = tot - left
    nl = best_i - lo + 1
    nr = n - nl
    h1 = _ent(left, nl)
    h2 = _ent(right, nr)
    k1 = 0
    k2 = 0
    for c in range(C):
        if left[c] > 0:
            k1 += 1
        if right[c] > 0:
            k2 += 1
    gain = hS - best_e
    delta = np.log2(3.0 ** k - 2) - (k * hS - k1 * h1 - k2 * h2)
    if gain <= (np.log2(n - 1) + delta) / n:
        return ncut
    cuts[ncut] = 0.5 * (xs[best_i] + xs[best_i + 1])
    ncut += 1
    ncut = _mdl_rec(xs, ys, lo, best_i + 1, C, cuts, ncut)
    ncut = _mdl_rec(xs, ys, best_i + 1, hi, C, cuts, ncut)
    return ncut

In [ ]:
def mdl_cutpoints(X, y, C):
    """Return list of sorted cut-point arrays, one per column of X."""
    out = []
    cuts = np.zeros(64)
    for j in range(X.shape[1]):
        o = np.argsort(X[:, j], kind="mergesort")
        nc = _mdl_rec(X[o, j].astype(np.float64), y[o].astype(np.int64), 0, len(y), C, cuts, 0)
        out.append(np.sort(cuts[:nc].copy()))
    return out

In [ ]:
def apply_cuts(X, cuts):
    D = np.zeros(X.shape, dtype=np.int64)
    for j, c in enumerate(cuts):
        D[:, j] = np.searchsorted(c, X[:, j], side="right")
    return D

### Entropy helpers

In [ ]:
# ---------------------------------------------------------------- entropy helpers
def _H_from_counts(cnt):
    cnt = cnt[cnt > 0]
    p = cnt / cnt.sum()
    return -(p * np.log2(p)).sum()

In [ ]:
def entropies(D, nb):
    """Marginal entropy of each discrete column."""
    n = D.shape[0]
    H = np.zeros(D.shape[1])
    for j in range(D.shape[1]):
        cnt = np.bincount(D[:, j], minlength=nb[j]).astype(float)
        H[j] = _H_from_counts(cnt)
    return H

In [ ]:
def su_vector(v, nbv, D, nb, Hv, H):
    """Symmetrical uncertainty between discrete vector v and every column of D."""
    n = D.shape[0]
    # joint code
    J = v[:, None] * nb.max() + D  # unique per pair as long as D < nb.max()
    su = np.zeros(D.shape[1])
    for j in range(D.shape[1]):
        cnt = np.bincount(J[:, j]).astype(float)
        hj = _H_from_counts(cnt)
        denom = Hv + H[j]
        su[j] = 0.0 if denom <= 0 else 2.0 * (Hv + H[j] - hj) / denom
    return su

In [ ]:
def su_vector_fast(v, D, nbmax, Hv, H):
    n, G = D.shape
    J = v[:, None] * nbmax + D
    K = int(J.max()) + 1
    # counts per (code, column) via offset bincount
    offs = J + (np.arange(G) * K)[None, :]
    cnt = np.bincount(offs.ravel(), minlength=G * K).reshape(G, K).astype(float)
    p = cnt / n
    with np.errstate(divide="ignore", invalid="ignore"):
        hj = -np.nansum(np.where(p > 0, p * np.log2(p), 0.0), axis=1)
    denom = Hv + H
    su = np.where(denom > 0, 2.0 * (Hv + H - hj) / np.where(denom > 0, denom, 1), 0.0)
    return su

### CFS with best-first search

In [ ]:
# ---------------------------------------------------------------- CFS with best-first search
def cfs_bfs(D, y, nb, max_stale=5):
    """Hall's CFS merit with WEKA-style forward best-first search.
    D: n x G discrete matrix (only informative columns), y: class codes."""
    n, G = D.shape
    nbmax = int(max(nb.max(), y.max() + 1))
    H = entropies(D, nb)
    Hy = _H_from_counts(np.bincount(y).astype(float))
    rcf = su_vector_fast(y, D, nbmax, Hy, H)
    su_rows = {}

    def row(g):
        if g not in su_rows:
            su_rows[g] = su_vector_fast(D[:, g], D, nbmax, H[g], H)
        return su_rows[g]

    def merit(k, s_cf, s_ff):
        return 0.0 if k == 0 else (s_cf) / np.sqrt(k + 2.0 * s_ff)

    best_set, best_m = (), 0.0
    visited = {frozenset()}
    heap = [(-0.0, 0, ())]
    stale = 0
    counter = 1
    while heap and stale < max_stale:
        negm, _, S = heapq.heappop(heap)
        k = len(S)
        s_cf = rcf[list(S)].sum() if k else 0.0
        sumrow = np.zeros(G)
        for g in S:
            sumrow += row(g)
        s_ff = 0.0
        if k > 1:
            Sl = list(S)
            s_ff = sum(row(a)[b] for i, a in enumerate(Sl) for b in Sl[i + 1:])
        child_m = (s_cf + rcf) / np.sqrt((k + 1) + 2.0 * (s_ff + sumrow))
        if k:
            child_m[list(S)] = -1
        order = np.argsort(-child_m)
        improved = False
        added = 0
        for g in order:
            if child_m[g] <= 0:
                break
            T = tuple(sorted(S + (int(g),)))
            fs = frozenset(T)
            if fs in visited:
                continue
            visited.add(fs)
            heapq.heappush(heap, (-child_m[g], counter, T))
            counter += 1
            added += 1
            if child_m[g] > best_m + 1e-12:
                best_m, best_set = float(child_m[g]), T
                improved = True
            if added >= 200:   # keep the open list bounded (top children only)
                break
        stale = 0 if improved else stale + 1
    return list(best_set), best_m

### Consistency with best-first search

In [ ]:
# ---------------------------------------------------------------- consistency with best-first search
def inconsistency_count(D, y, C):
    if D.shape[1] == 0:
        return len(y) - np.bincount(y, minlength=C).max()
    _, inv = np.unique(D, axis=0, return_inverse=True)
    inv = inv.ravel()
    P = inv.max() + 1
    cnt = np.zeros((P, C))
    np.add.at(cnt, (inv, y), 1)
    return (cnt.sum(1) - cnt.max(1)).sum()

In [ ]:
def consistency_bfs(D, y, C, max_stale=5):
    """Liu & Setiono consistency measure with WEKA-style forward best-first search.
    Returns the smallest subset found whose consistency equals the best reached."""
    n, G = D.shape
    full_cons = 1 - inconsistency_count(D, y, C) / n
    best_set, best_c = (), 1 - inconsistency_count(D[:, []], y, C) / n
    visited = {frozenset()}
    heap = [(-best_c, 0, ())]
    stale, counter = 0, 1
    while heap and stale < max_stale:
        _, _, S = heapq.heappop(heap)
        improved = False
        for g in range(G):
            if g in S:
                continue
            T = tuple(sorted(S + (g,)))
            fs = frozenset(T)
            if fs in visited:
                continue
            visited.add(fs)
            c = 1 - inconsistency_count(D[:, list(T)], y, C) / n
            heapq.heappush(heap, (-c, counter, T))
            counter += 1
            if c > best_c + 1e-12:
                best_c, best_set, improved = c, T, True
        stale = 0 if improved else stale + 1
        if best_c >= full_cons - 1e-12:
            # WEKA keeps searching until stale; the min-features subset is the first that reaches it
            pass
    return list(best_set), best_c, full_cons

### Bayesian classifiers

In [ ]:
# ---------------------------------------------------------------- Bayesian classifiers
class HNB:
    """Hidden naive Bayes (Zhang, Jiang & Su 2005), as in WEKA's HNB."""

    def fit(self, D, y, nb, C):
        self.C, self.nb = C, np.asarray(nb)
        n, m = D.shape
        self.m = m
        self.prior = (np.bincount(y, minlength=C) + 1.0) / (n + C)
        # counts N(ai, aj, c)
        self.cond = {}
        self.single = []
        for i in range(m):
            t = np.zeros((self.nb[i], C))
            np.add.at(t, (D[:, i], y), 1)
            self.single.append((t + 1.0) / (t.sum(0, keepdims=True) + self.nb[i]))
        W = np.zeros((m, m))
        for i in range(m):
            for j in range(m):
                if i == j:
                    continue
                t = np.zeros((self.nb[i], self.nb[j], C))
                np.add.at(t, (D[:, i], D[:, j], y), 1)
                # P(ai | aj, c) with Laplace correction
                self.cond[(i, j)] = (t + 1.0) / (t.sum(0, keepdims=True) + self.nb[i])
                # conditional mutual information I(Ai;Aj|C)
                p = t / n
                pic = p.sum(1, keepdims=True)
                pjc = p.sum(0, keepdims=True)
                pc = p.sum((0, 1), keepdims=True)
                with np.errstate(divide="ignore", invalid="ignore"):
                    term = np.where(p > 0, p * np.log(p * pc / (pic * pjc)), 0.0)
                W[i, j] = np.nansum(term)
        for i in range(m):
            s = W[i].sum()
            W[i] = W[i] / s if s > 0 else (np.where(np.arange(m) != i, 1.0 / max(m - 1, 1), 0))
        self.W = W
        return self

    def predict_proba(self, D):
        D = np.minimum(D, self.nb[None, :] - 1)
        n, m = D.shape
        logp = np.tile(np.log(self.prior), (n, 1))
        for i in range(m):
            if m == 1:
                logp += np.log(self.single[i][D[:, i]])
                continue
            acc = np.zeros((n, self.C))
            for j in range(m):
                if i == j:
                    continue
                acc += self.W[i, j] * self.cond[(i, j)][D[:, i], D[:, j]]
            logp += np.log(acc)
        logp -= logp.max(1, keepdims=True)
        p = np.exp(logp)
        return p / p.sum(1, keepdims=True)

    def predict(self, D):
        return self.predict_proba(D).argmax(1)

In [ ]:
class NB:
    def fit(self, D, y, nb, C, weights=None):
        self.C, self.nb = C, np.asarray(nb)
        n, m = D.shape
        self.prior = (np.bincount(y, minlength=C) + 1.0) / (n + C)
        self.tab = []
        for i in range(m):
            t = np.zeros((self.nb[i], C))
            np.add.at(t, (D[:, i], y), 1)
            self.tab.append((t + 1.0) / (t.sum(0, keepdims=True) + self.nb[i]))
        self.w = np.ones(m) if weights is None else weights
        return self

    def predict_proba(self, D):
        D = np.minimum(D, self.nb[None, :] - 1)
        logp = np.tile(np.log(self.prior), (D.shape[0], 1))
        for i in range(D.shape[1]):
            logp += self.w[i] * np.log(self.tab[i][D[:, i]])
        logp -= logp.max(1, keepdims=True)
        p = np.exp(logp)
        return p / p.sum(1, keepdims=True)

    def predict(self, D):
        return self.predict_proba(D).argmax(1)

In [ ]:
def _mi(a, b):
    ja = a * (b.max() + 1) + b
    return (_H_from_counts(np.bincount(a).astype(float)) + _H_from_counts(np.bincount(b).astype(float))
            - _H_from_counts(np.bincount(ja).astype(float)))

In [ ]:
class CWNB(NB):
    """Correlation-based feature weighting naive Bayes (Jiang et al., TKDE 2019):
    w_i = sigmoid(relevance_i - mean redundancy_i), using normalised mutual information."""

    def fit(self, D, y, nb, C):
        m = D.shape[1]
        Hy = _H_from_counts(np.bincount(y).astype(float))
        Hs = [_H_from_counts(np.bincount(D[:, i]).astype(float)) for i in range(m)]
        rel = np.array([2 * _mi(D[:, i], y) / (Hs[i] + Hy) if Hs[i] + Hy > 0 else 0 for i in range(m)])
        red = np.zeros(m)
        for i in range(m):
            vals = [2 * _mi(D[:, i], D[:, j]) / (Hs[i] + Hs[j]) if Hs[i] + Hs[j] > 0 else 0
                    for j in range(m) if j != i]
            red[i] = np.mean(vals) if vals else 0.0
        w = 1.0 / (1.0 + np.exp(-(rel - red)))
        return super().fit(D, y, nb, C, weights=w)